In [2]:
import os
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import optuna
from datetime import datetime

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

# --- KONFIGURASI FOLDER KHUSUS TUNING ---
DATASET_DIR = pathlib.Path("data_cropped") # Tetap membaca dari sumber yang sama
TUNING_DIR = "HASIL_TUNING_BAYESIAN"
LOG_FILE = os.path.join(TUNING_DIR, "log_eksperimen_tuning04.csv")
IMG_SIZE = 224

# Buat folder khusus tuning jika belum ada
os.makedirs(TUNING_DIR, exist_ok=True)

# Pemuatan Path Data
all_image_paths = [str(p) for p in DATASET_DIR.glob('*/*.png')]
class_names_list = sorted([item.name for item in DATASET_DIR.glob('*') if item.is_dir()])
label_to_index = dict((name, index) for index, name in enumerate(class_names_list))
all_image_labels = [label_to_index[pathlib.Path(p).parent.name] for p in all_image_paths]

X = np.array(all_image_paths)
y = np.array(all_image_labels)

# --- FUNGSI PIPELINE DATA ---
def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.io.decode_image(img, channels=4, expand_animations=False)
    img = tf.cast(img, tf.float32)
    rgb = img[..., :3]
    alpha = img[..., 3:4] / 255.0
    img_fixed = rgb * alpha
    img_resized = tf.image.resize(img_fixed, [IMG_SIZE, IMG_SIZE])
    img_final = img_resized / 255.0
    img_final.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img_final, tf.one_hot(label, depth=len(class_names_list))

# --- FUNGSI BUILD MODEL (MODIFIKASI CORONG) ---
def build_model(dense_units, dropout_rate, use_corong):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    max_p = layers.GlobalMaxPooling2D()(x)
    avg_p = layers.GlobalAveragePooling2D()(x)
    merged = layers.Concatenate()([max_p, avg_p])
    
    # LOGIKA PENGGUNAAN CORONG
    if use_corong:
        x = layers.Dense(dense_units, activation='relu')(merged)
        x = layers.Dropout(dropout_rate)(x)
        outputs = layers.Dense(4, activation='softmax')(x)
    else:
        outputs = layers.Dense(4, activation='softmax')(merged)
    
    return models.Model(inputs=inputs, outputs=outputs)

# --- FUNGSI TUJUAN OPTUNA (OBJECTIVE) ---
def objective(trial):
    # 1. Optuna Memilih Angka Kombinasi
    use_corong = trial.suggest_categorical("use_corong", [True, False])
    focal_loss_gamma = trial.suggest_categorical("focal_loss_gamma", [0, 1, 2]) 
    lr = trial.suggest_categorical("learning_rate", [5e-5, 1e-4, 5e-4])
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    dense_units = trial.suggest_categorical("dense_units", [128, 256])
    dropout_rate = trial.suggest_categorical("dropout_rate", [0.2, 0.3, 0.4])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])

    print(f"\n" + "-"*50)
    print(f"▶️ MEMULAI TRIAL KE-{trial.number}")
    print(f"⚙️ Parameter: Corong={use_corong}, Gamma={focal_loss_gamma}, LR={lr}, Batch={batch_size}, Dense={dense_units}, Dropout={dropout_rate}, Opt={optimizer_name}")
    print("-"*50)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_accs = []

    # 2. Uji Kombinasi ke dalam 5 Lipatan (K-Fold)
    for fold_no, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # ==============================================================================
        # --- TAMBAHAN BARU: OVERSAMPLING ON-THE-FLY KHUSUS DATA LATIH TUNING ---
        # ==============================================================================
        class_4_indices = np.where(y_train == 3)[0]
        X_train_class_4 = X_train[class_4_indices]
        y_train_class_4 = y_train[class_4_indices]
        
        # Duplikasi fisik 2x lipat di memori
        X_train = np.concatenate([X_train, X_train_class_4])
        y_train = np.concatenate([y_train, y_train_class_4])
        
        # Pengacakan urutan data latih
        shuffle_idx = np.random.permutation(len(X_train))
        X_train = X_train[shuffle_idx]
        y_train = y_train[shuffle_idx]
        # ==============================================================================

        # Pipeline Dataset menggunakan tf.data
        train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).map(process_path).shuffle(len(X_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
        val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).map(process_path).batch(batch_size).prefetch(tf.data.AUTOTUNE)

        tf.keras.backend.clear_session()
        
        model = build_model(dense_units, dropout_rate, use_corong)
        
        if optimizer_name == "Adam":
            opt = Adam(learning_rate=lr)
        else:            
            opt = RMSprop(learning_rate=lr)

        # ==============================================================================
        # --- PENENTUAN FUNGSI LOSS BERSIH (TANPA CLASS WEIGHT) + LABEL SMOOTHING 0.1 ---
        # ==============================================================================
        if focal_loss_gamma == 0:
            # Menggunakan Categorical Crossentropy + Label Smoothing 0.1
            loss_function = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
        else:
            # Menggunakan Focal Loss murni + Label Smoothing 0.1 (Tanpa parameter alpha!)
            loss_function = tf.keras.losses.CategoricalFocalCrossentropy(
                gamma=float(focal_loss_gamma),
                label_smoothing=0.1
            )
        # ==============================================================================

        model.compile(optimizer=opt, loss=loss_function, metrics=['accuracy'])

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True), 
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6) 
        ]

        # Proses Pelatihan (Tanpa menyertakan parameter class_weight)
        model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks, verbose=0)
        
        y_val_true = np.concatenate([lbl for img, lbl in val_ds], axis=0)
        y_val_pred = model.predict(val_ds, verbose=0)
        acc = accuracy_score(np.argmax(y_val_true, axis=1), np.argmax(y_val_pred, axis=1))
        fold_accs.append(acc)
        
        print(f"    ✓ Fold {fold_no} Selesai | Akurasi: {acc*100:.2f}%")

    # 3. Dapatkan Rata-rata Akurasi dari 5 Lipatan
    avg_accuracy = np.mean(fold_accs)
    
    print(f"🏁 KESIMPULAN TRIAL {trial.number} | Rata-rata Akurasi: {avg_accuracy*100:.2f}%")
    
    # --- PENCATATAN LOG KE CSV ---
    log_data = {
        "Trial_No": trial.number,
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Use_Corong": use_corong,
        "Focal_Loss_Gamma": focal_loss_gamma, 
        "Learning_Rate": lr,
        "Batch_Size": batch_size,
        "Dense_Units": dense_units,
        "Dropout_Rate": dropout_rate,
        "Optimizer": optimizer_name,
        "Avg_Accuracy": round(avg_accuracy * 100, 2)
    }
    df_log = pd.DataFrame([log_data])
    if not os.path.exists(LOG_FILE):
        df_log.to_csv(LOG_FILE, index=False)
    else:
        df_log.to_csv(LOG_FILE, mode='a', header=False, index=False)

    return avg_accuracy

# --- EKSEKUSI TUNING ---
print("\n[INFO] Memulai Proses Hyperparameter Tuning...")
print(f"[INFO] Hasil akan dicatat secara real-time di: {LOG_FILE}\n")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("\n" + "="*60)
print("🏆 HASIL TUNING TERBAIK DITEMUKAN!")
print(f"  Akurasi Rata-rata Tertinggi: {study.best_value*100:.2f}%")
print(f"  Coba Parameter Ini di Script Final Anda:")
for key, value in study.best_params.items():
    print(f"    - {key}: {value}")
print("="*60)

[I 2026-06-25 18:55:18,212] A new study created in memory with name: no-name-df85baa1-e26d-45c7-9c9f-8edbfba9881e



[INFO] Memulai Proses Hyperparameter Tuning...
[INFO] Hasil akan dicatat secara real-time di: HASIL_TUNING_BAYESIAN\log_eksperimen_tuning04.csv


--------------------------------------------------
▶️ MEMULAI TRIAL KE-0
⚙️ Parameter: Corong=False, Gamma=0, LR=0.0001, Batch=32, Dense=256, Dropout=0.2, Opt=Adam
--------------------------------------------------
    ✓ Fold 1 Selesai | Akurasi: 46.15%
    ✓ Fold 2 Selesai | Akurasi: 36.81%
    ✓ Fold 3 Selesai | Akurasi: 10.44%
    ✓ Fold 4 Selesai | Akurasi: 41.99%


[I 2026-06-25 19:39:45,996] Trial 0 finished with value: 0.35366401554246857 and parameters: {'use_corong': False, 'focal_loss_gamma': 0, 'learning_rate': 0.0001, 'batch_size': 32, 'dense_units': 256, 'dropout_rate': 0.2, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.35366401554246857.


    ✓ Fold 5 Selesai | Akurasi: 41.44%
🏁 KESIMPULAN TRIAL 0 | Rata-rata Akurasi: 35.37%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-1
⚙️ Parameter: Corong=False, Gamma=2, LR=0.0001, Batch=16, Dense=256, Dropout=0.3, Opt=Adam
--------------------------------------------------
    ✓ Fold 1 Selesai | Akurasi: 14.29%
    ✓ Fold 2 Selesai | Akurasi: 26.37%
    ✓ Fold 3 Selesai | Akurasi: 41.76%
    ✓ Fold 4 Selesai | Akurasi: 25.97%


[I 2026-06-25 20:24:05,762] Trial 1 finished with value: 0.299641794669419 and parameters: {'use_corong': False, 'focal_loss_gamma': 2, 'learning_rate': 0.0001, 'batch_size': 16, 'dense_units': 256, 'dropout_rate': 0.3, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.35366401554246857.


...

    ✓ Fold 5 Selesai | Akurasi: 51.38%
🏁 KESIMPULAN TRIAL 17 | Rata-rata Akurasi: 50.33%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-18
⚙️ Parameter: Corong=True, Gamma=0, LR=0.0005, Batch=32, Dense=128, Dropout=0.4, Opt=Adam
--------------------------------------------------
    ✓ Fold 1 Selesai | Akurasi: 52.20%
    ✓ Fold 2 Selesai | Akurasi: 52.75%
    ✓ Fold 3 Selesai | Akurasi: 49.45%
    ✓ Fold 4 Selesai | Akurasi: 57.46%


[I 2026-06-26 08:50:45,489] Trial 18 finished with value: 0.5353105458077835 and parameters: {'use_corong': True, 'focal_loss_gamma': 0, 'learning_rate': 0.0005, 'batch_size': 32, 'dense_units': 128, 'dropout_rate': 0.4, 'optimizer': 'Adam'}. Best is trial 18 with value: 0.5353105458077835.


    ✓ Fold 5 Selesai | Akurasi: 55.80%
🏁 KESIMPULAN TRIAL 18 | Rata-rata Akurasi: 53.53%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-19
⚙️ Parameter: Corong=True, Gamma=0, LR=0.0005, Batch=32, Dense=128, Dropout=0.4, Opt=Adam
--------------------------------------------------
    ✓ Fold 1 Selesai | Akurasi: 48.90%
    ✓ Fold 2 Selesai | Akurasi: 46.70%
    ✓ Fold 3 Selesai | Akurasi: 54.40%
    ✓ Fold 4 Selesai | Akurasi: 53.04%


[I 2026-06-26 09:30:21,622] Trial 19 finished with value: 0.5143646408839778 and parameters: {'use_corong': True, 'focal_loss_gamma': 0, 'learning_rate': 0.0005, 'batch_size': 32, 'dense_units': 128, 'dropout_rate': 0.4, 'optimizer': 'Adam'}. Best is trial 18 with value: 0.5353105458077835.


    ✓ Fold 5 Selesai | Akurasi: 54.14%
🏁 KESIMPULAN TRIAL 19 | Rata-rata Akurasi: 51.44%

🏆 HASIL TUNING TERBAIK DITEMUKAN!
  Akurasi Rata-rata Tertinggi: 53.53%
  Coba Parameter Ini di Script Final Anda:
    - use_corong: True
    - focal_loss_gamma: 0
    - learning_rate: 0.0005
    - batch_size: 32
    - dense_units: 128
    - dropout_rate: 0.4
    - optimizer: Adam


In [ ]:
# import os
# import pathlib
# import numpy as np
# import pandas as pd
# import tensorflow as tf
# import optuna
# from datetime import datetime

# from tensorflow.keras import layers, models
# from tensorflow.keras.optimizers import Adam,RMSprop
# from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# from sklearn.metrics import accuracy_score
# from sklearn.model_selection import StratifiedKFold
# from sklearn.utils import class_weight

# # --- KONFIGURASI FOLDER KHUSUS TUNING ---
# DATASET_DIR = pathlib.Path("data_cropped") # Tetap membaca dari sumber yang sama
# TUNING_DIR = "HASIL_TUNING_OPTUNA"
# LOG_FILE = os.path.join(TUNING_DIR, "log_eksperimen_tuning05.csv")
# IMG_SIZE = 224

# # Buat folder khusus tuning jika belum ada
# os.makedirs(TUNING_DIR, exist_ok=True)

# # Pemuatan Path Data
# all_image_paths = [str(p) for p in DATASET_DIR.glob('*/*.png')]
# class_names_list = sorted([item.name for item in DATASET_DIR.glob('*') if item.is_dir()])
# label_to_index = dict((name, index) for index, name in enumerate(class_names_list))
# all_image_labels = [label_to_index[pathlib.Path(p).parent.name] for p in all_image_paths]

# X = np.array(all_image_paths)
# y = np.array(all_image_labels)

# # --- FUNGSI PIPELINE DATA ---
# def process_path(file_path, label):
#     img = tf.io.read_file(file_path)
#     img = tf.io.decode_image(img, channels=4, expand_animations=False)
#     img = tf.cast(img, tf.float32)
#     rgb = img[..., :3]
#     alpha = img[..., 3:4] / 255.0
#     img_fixed = rgb * alpha
#     img_resized = tf.image.resize(img_fixed, [IMG_SIZE, IMG_SIZE])
#     img_final = img_resized / 255.0
#     img_final.set_shape([IMG_SIZE, IMG_SIZE, 3])
#     return img_final, tf.one_hot(label, depth=len(class_names_list))

# # --- FUNGSI BUILD MODEL (MODIFIKASI CORONG) ---
# def build_model(dense_units, dropout_rate, use_corong):
#     inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
#     x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     max_p = layers.GlobalMaxPooling2D()(x)
#     avg_p = layers.GlobalAveragePooling2D()(x)
#     merged = layers.Concatenate()([max_p, avg_p])
    
#     # LOGIKA PENGGUNAAN CORONG
#     if use_corong:
#         # Jika True: Fitur masuk ke corong Dense + Dropout dulu
#         x = layers.Dense(dense_units, activation='relu')(merged)
#         x = layers.Dropout(dropout_rate)(x)
#         outputs = layers.Dense(4, activation='softmax')(x)
#     else:
#         # Jika False: Fitur langsung menabrak lapisan klasifikasi (Ablation)
#         outputs = layers.Dense(4, activation='softmax')(merged)
    
#     return models.Model(inputs=inputs, outputs=outputs)

# # --- FUNGSI TUJUAN OPTUNA (OBJECTIVE) ---
# def objective(trial):
#     # 1. Optuna Memilih Angka Kombinasi
#     use_corong = trial.suggest_categorical("use_corong", [True, False])
#     focal_loss_gamma = trial.suggest_categorical("focal_loss_gamma", [0, 1, 2]) # Penambahan Opsi Focal Loss
#     lr = trial.suggest_categorical("learning_rate", [5e-5, 1e-4, 5e-4])
#     batch_size = trial.suggest_categorical("batch_size", [16, 32])
#     dense_units = trial.suggest_categorical("dense_units", [128, 256])
#     dropout_rate = trial.suggest_categorical("dropout_rate", [0.2, 0.3, 0.4])
#     optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])

#     # --- TAMBAHAN PRINT INFO TRIAL ---
#     print(f"\n" + "-"*50)
#     print(f"▶️ MEMULAI TRIAL KE-{trial.number}")
#     print(f"⚙️ Parameter: Corong={use_corong}, Gamma={focal_loss_gamma}, LR={lr}, Batch={batch_size}, Dense={dense_units}, Dropout={dropout_rate}, Opt={optimizer_name}")
#     print("-"*50)

#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     fold_accs = []

#     # 2. Uji Kombinasi ke dalam 5 Lipatan (K-Fold)
#     for fold_no, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#         X_train, X_val = X[train_idx], X[val_idx]
#         y_train, y_val = y[train_idx], y[val_idx]

#         weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
#         alpha_list = [float(w) for w in weights]

#         train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).map(process_path).shuffle(len(X_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
#         val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).map(process_path).batch(batch_size).prefetch(tf.data.AUTOTUNE)

#         tf.keras.backend.clear_session()
        
#         # Meneruskan pilihan corong ke fungsi build_model
#         model = build_model(dense_units, dropout_rate, use_corong)
        
#         if optimizer_name == "Adam":
#             opt = Adam(learning_rate=lr)
#         else:            
#             opt = RMSprop(learning_rate=lr)

#         # PENENTUAN FUNGSI LOSS BERDASARKAN NILAI GAMMA
#         if focal_loss_gamma == 0:
#             # Gamma 0 secara matematis sama dengan CCE standar
#             loss_function = 'categorical_crossentropy'
#         else:
#             # Gamma 1 atau 2 mengaktifkan Focal Loss dengan pembobotan kelas
#             loss_function = tf.keras.losses.CategoricalFocalCrossentropy(
#                 alpha=alpha_list, 
#                 gamma=float(focal_loss_gamma)
#             )

#         model.compile(optimizer=opt, loss=loss_function, metrics=['accuracy'])

#         callbacks = [
#             EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True), 
#             ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6) 
#         ]

#         model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks, verbose=0)
        
#         y_val_true = np.concatenate([lbl for img, lbl in val_ds], axis=0)
#         y_val_pred = model.predict(val_ds, verbose=0)
#         acc = accuracy_score(np.argmax(y_val_true, axis=1), np.argmax(y_val_pred, axis=1))
#         fold_accs.append(acc)
        
#         # --- TAMBAHAN PRINT INFO PER FOLD ---
#         print(f"   ✓ Fold {fold_no} Selesai | Akurasi: {acc*100:.2f}%")

#     # 3. Dapatkan Rata-rata Akurasi dari 5 Lipatan
#     avg_accuracy = np.mean(fold_accs)
    
#     # --- TAMBAHAN PRINT KESIMPULAN TRIAL ---
#     print(f"🏁 KESIMPULAN TRIAL {trial.number} | Rata-rata Akurasi: {avg_accuracy*100:.2f}%")
    
#     # --- PENCATATAN LOG KE CSV ---
#     log_data = {
#         "Trial_No": trial.number,
#         "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#         "Use_Corong": use_corong,
#         "Focal_Loss_Gamma": focal_loss_gamma, # Ditambahkan ke pencatatan CSV
#         "Learning_Rate": lr,
#         "Batch_Size": batch_size,
#         "Dense_Units": dense_units,
#         "Dropout_Rate": dropout_rate,
#         "Optimizer": optimizer_name,
#         "Avg_Accuracy": round(avg_accuracy * 100, 2)
#     }
#     df_log = pd.DataFrame([log_data])
#     if not os.path.exists(LOG_FILE):
#         df_log.to_csv(LOG_FILE, index=False)
#     else:
#         df_log.to_csv(LOG_FILE, mode='a', header=False, index=False)

#     return avg_accuracy

# # --- EKSEKUSI TUNING ---
# print("\n[INFO] Memulai Proses Hyperparameter Tuning...")
# print(f"[INFO] Hasil akan dicatat secara real-time di: {LOG_FILE}\n")

# study = optuna.create_study(direction="maximize")
# # Anda bisa menaikkan n_trials jika ingin eksplorasi lebih dalam
# study.optimize(objective, n_trials=20)

# print("\n" + "="*60)
# print("🏆 HASIL TUNING TERBAIK DITEMUKAN!")
# print(f"  Akurasi Rata-rata Tertinggi: {study.best_value*100:.2f}%")
# print(f"  Coba Parameter Ini di Script Final Anda:")
# for key, value in study.best_params.items():
#     print(f"    - {key}: {value}")
# print("="*60)

[I 2026-06-05 10:46:14,607] A new study created in memory with name: no-name-91ad4cbb-18d1-4484-9d37-f0993ffa1a05



[INFO] Memulai Proses Hyperparameter Tuning...
[INFO] Hasil akan dicatat secara real-time di: HASIL_TUNING_OPTUNA\log_eksperimen_tuning05.csv


--------------------------------------------------
▶️ MEMULAI TRIAL KE-0
⚙️ Parameter: Corong=True, Gamma=2, LR=0.0001, Batch=16, Dense=256, Dropout=0.2, Opt=RMSprop
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 33.52%
   ✓ Fold 2 Selesai | Akurasi: 49.45%
   ✓ Fold 3 Selesai | Akurasi: 50.55%
   ✓ Fold 4 Selesai | Akurasi: 48.62%


[I 2026-06-05 11:49:40,753] Trial 0 finished with value: 0.45045838139760785 and parameters: {'use_corong': True, 'focal_loss_gamma': 2, 'learning_rate': 0.0001, 'batch_size': 16, 'dense_units': 256, 'dropout_rate': 0.2, 'optimizer': 'RMSprop'}. Best is trial 0 with value: 0.45045838139760785.


   ✓ Fold 5 Selesai | Akurasi: 43.09%
🏁 KESIMPULAN TRIAL 0 | Rata-rata Akurasi: 45.05%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-1
⚙️ Parameter: Corong=True, Gamma=2, LR=0.0001, Batch=16, Dense=128, Dropout=0.2, Opt=RMSprop
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 40.66%
   ✓ Fold 2 Selesai | Akurasi: 39.56%
   ✓ Fold 3 Selesai | Akurasi: 50.00%
   ✓ Fold 4 Selesai | Akurasi: 38.12%


[I 2026-06-05 12:49:42,709] Trial 1 finished with value: 0.4239754720417704 and parameters: {'use_corong': True, 'focal_loss_gamma': 2, 'learning_rate': 0.0001, 'batch_size': 16, 'dense_units': 128, 'dropout_rate': 0.2, 'optimizer': 'RMSprop'}. Best is trial 0 with value: 0.45045838139760785.


...

   ✓ Fold 5 Selesai | Akurasi: 26.52%
🏁 KESIMPULAN TRIAL 17 | Rata-rata Akurasi: 41.40%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-18
⚙️ Parameter: Corong=False, Gamma=0, LR=0.0005, Batch=32, Dense=128, Dropout=0.3, Opt=Adam
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 21.43%
   ✓ Fold 2 Selesai | Akurasi: 45.05%
   ✓ Fold 3 Selesai | Akurasi: 46.15%
   ✓ Fold 4 Selesai | Akurasi: 45.30%


[I 2026-06-06 03:32:42,926] Trial 18 finished with value: 0.3976504158824601 and parameters: {'use_corong': False, 'focal_loss_gamma': 0, 'learning_rate': 0.0005, 'batch_size': 32, 'dense_units': 128, 'dropout_rate': 0.3, 'optimizer': 'Adam'}. Best is trial 6 with value: 0.5021978021978022.


   ✓ Fold 5 Selesai | Akurasi: 40.88%
🏁 KESIMPULAN TRIAL 18 | Rata-rata Akurasi: 39.77%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-19
⚙️ Parameter: Corong=True, Gamma=0, LR=0.0005, Batch=32, Dense=128, Dropout=0.4, Opt=RMSprop
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 48.90%
   ✓ Fold 2 Selesai | Akurasi: 52.20%
   ✓ Fold 3 Selesai | Akurasi: 49.45%
   ✓ Fold 4 Selesai | Akurasi: 51.38%


[I 2026-06-06 04:10:21,689] Trial 19 finished with value: 0.5099386801044259 and parameters: {'use_corong': True, 'focal_loss_gamma': 0, 'learning_rate': 0.0005, 'batch_size': 32, 'dense_units': 128, 'dropout_rate': 0.4, 'optimizer': 'RMSprop'}. Best is trial 19 with value: 0.5099386801044259.


   ✓ Fold 5 Selesai | Akurasi: 53.04%
🏁 KESIMPULAN TRIAL 19 | Rata-rata Akurasi: 50.99%

🏆 HASIL TUNING TERBAIK DITEMUKAN!
  Akurasi Rata-rata Tertinggi: 50.99%
  Coba Parameter Ini di Script Final Anda:
    - use_corong: True
    - focal_loss_gamma: 0
    - learning_rate: 0.0005
    - batch_size: 32
    - dense_units: 128
    - dropout_rate: 0.4
    - optimizer: RMSprop


In [ ]:
# import os
# import pathlib
# import numpy as np
# import pandas as pd
# import tensorflow as tf
# import optuna
# from datetime import datetime

# from tensorflow.keras import layers, models
# from tensorflow.keras.optimizers import Adam,RMSprop
# from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# from sklearn.metrics import accuracy_score
# from sklearn.model_selection import StratifiedKFold
# from sklearn.utils import class_weight

# # --- KONFIGURASI FOLDER KHUSUS TUNING ---
# DATASET_DIR = pathlib.Path("data_cropped") # Tetap membaca dari sumber yang sama
# TUNING_DIR = "HASIL_TUNING_OPTUNA02"
# LOG_FILE = os.path.join(TUNING_DIR, "log_eksperimen_tuning02.csv")
# IMG_SIZE = 224

# # Buat folder khusus tuning jika belum ada
# os.makedirs(TUNING_DIR, exist_ok=True)

# # Pemuatan Path Data
# all_image_paths = [str(p) for p in DATASET_DIR.glob('*/*.png')]
# class_names_list = sorted([item.name for item in DATASET_DIR.glob('*') if item.is_dir()])
# label_to_index = dict((name, index) for index, name in enumerate(class_names_list))
# all_image_labels = [label_to_index[pathlib.Path(p).parent.name] for p in all_image_paths]

# X = np.array(all_image_paths)
# y = np.array(all_image_labels)

# # --- FUNGSI PIPELINE DATA ---
# def process_path(file_path, label):
#     img = tf.io.read_file(file_path)
#     img = tf.io.decode_image(img, channels=4, expand_animations=False)
#     img = tf.cast(img, tf.float32)
#     rgb = img[..., :3]
#     alpha = img[..., 3:4] / 255.0
#     img_fixed = rgb * alpha
#     img_resized = tf.image.resize(img_fixed, [IMG_SIZE, IMG_SIZE])
#     img_final = img_resized / 255.0
#     img_final.set_shape([IMG_SIZE, IMG_SIZE, 3])
#     return img_final, tf.one_hot(label, depth=len(class_names_list))

# # --- FUNGSI BUILD MODEL (MODIFIKASI CORONG) ---
# def build_model(dense_units, dropout_rate, use_corong):
#     inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
#     x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     max_p = layers.GlobalMaxPooling2D()(x)
#     avg_p = layers.GlobalAveragePooling2D()(x)
#     merged = layers.Concatenate()([max_p, avg_p])
    
#     # LOGIKA PENGGUNAAN CORONG
#     if use_corong:
#         # Jika True: Fitur masuk ke corong Dense + Dropout dulu
#         x = layers.Dense(dense_units, activation='relu')(merged)
#         x = layers.Dropout(dropout_rate)(x)
#         outputs = layers.Dense(4, activation='softmax')(x)
#     else:
#         # Jika False: Fitur langsung menabrak lapisan klasifikasi (Ablation)
#         outputs = layers.Dense(4, activation='softmax')(merged)
    
#     return models.Model(inputs=inputs, outputs=outputs)

# # --- FUNGSI TUJUAN OPTUNA (OBJECTIVE) ---
# def objective(trial):
#     # 1. Optuna Memilih Angka Kombinasi
#     use_corong = trial.suggest_categorical("use_corong", [True, False]) # Penambahan Opsi Corong
#     lr = trial.suggest_categorical("learning_rate", [5e-5, 1e-4, 5e-4])
#     batch_size = trial.suggest_categorical("batch_size", [16, 32])
#     dense_units = trial.suggest_categorical("dense_units", [128, 256])
#     dropout_rate = trial.suggest_categorical("dropout_rate", [0.2, 0.3, 0.4])
#     optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])

#     # --- TAMBAHAN PRINT INFO TRIAL ---
#     print(f"\n" + "-"*50)
#     print(f"▶️ MEMULAI TRIAL KE-{trial.number}")
#     print(f"⚙️ Parameter: Corong={use_corong}, LR={lr}, Batch={batch_size}, Dense={dense_units}, Dropout={dropout_rate}, Opt={optimizer_name}")
#     print("-"*50)

#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     fold_accs = []

#     # 2. Uji Kombinasi ke dalam 5 Lipatan (K-Fold)
#     for fold_no, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#         X_train, X_val = X[train_idx], X[val_idx]
#         y_train, y_val = y[train_idx], y[val_idx]

#         weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
#         alpha_list = [float(w) for w in weights]

#         train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).map(process_path).shuffle(len(X_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
#         val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).map(process_path).batch(batch_size).prefetch(tf.data.AUTOTUNE)

#         tf.keras.backend.clear_session()
        
#         # Meneruskan pilihan corong ke fungsi build_model
#         model = build_model(dense_units, dropout_rate, use_corong)
        
#         if optimizer_name == "Adam":
#             opt = Adam(learning_rate=lr)
#         else:            
#             opt = RMSprop(learning_rate=lr)

#         model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

#         callbacks = [
#             EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True), 
#             ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6) 
#         ]

#         model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks, verbose=0)
        
#         y_val_true = np.concatenate([lbl for img, lbl in val_ds], axis=0)
#         y_val_pred = model.predict(val_ds, verbose=0)
#         acc = accuracy_score(np.argmax(y_val_true, axis=1), np.argmax(y_val_pred, axis=1))
#         fold_accs.append(acc)
        
#         # --- TAMBAHAN PRINT INFO PER FOLD ---
#         print(f"   ✓ Fold {fold_no} Selesai | Akurasi: {acc*100:.2f}%")

#     # 3. Dapatkan Rata-rata Akurasi dari 5 Lipatan
#     avg_accuracy = np.mean(fold_accs)
    
#     # --- TAMBAHAN PRINT KESIMPULAN TRIAL ---
#     print(f"🏁 KESIMPULAN TRIAL {trial.number} | Rata-rata Akurasi: {avg_accuracy*100:.2f}%")
    
#     # --- PENCATATAN LOG KE CSV ---
#     log_data = {
#         "Trial_No": trial.number,
#         "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#         "Use_Corong": use_corong, # Ditambahkan ke pencatatan CSV
#         "Learning_Rate": lr,
#         "Batch_Size": batch_size,
#         "Dense_Units": dense_units,
#         "Dropout_Rate": dropout_rate,
#         "Optimizer": optimizer_name,
#         "Avg_Accuracy": round(avg_accuracy * 100, 2)
#     }
#     df_log = pd.DataFrame([log_data])
#     if not os.path.exists(LOG_FILE):
#         df_log.to_csv(LOG_FILE, index=False)
#     else:
#         df_log.to_csv(LOG_FILE, mode='a', header=False, index=False)

#     return avg_accuracy

# # --- EKSEKUSI TUNING ---
# print("\n[INFO] Memulai Proses Hyperparameter Tuning...")
# print(f"[INFO] Hasil akan dicatat secara real-time di: {LOG_FILE}\n")

# study = optuna.create_study(direction="maximize")
# # Anda bisa menaikkan n_trials jika ingin eksplorasi lebih dalam
# study.optimize(objective, n_trials=20)

# print("\n" + "="*60)
# print("🏆 HASIL TUNING TERBAIK DITEMUKAN!")
# print(f"  Akurasi Rata-rata Tertinggi: {study.best_value*100:.2f}%")
# print(f"  Coba Parameter Ini di Script Final Anda:")
# for key, value in study.best_params.items():
#     print(f"    - {key}: {value}")
# print("="*60)

c:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-13 10:33:17,923] A new study created in memory with name: no-name-6c0ebfbe-e70b-4fca-9eeb-cb8903ae2e4a



[INFO] Memulai Proses Hyperparameter Tuning...
[INFO] Hasil akan dicatat secara real-time di: HASIL_TUNING_OPTUNA02\log_eksperimen_tuning02.csv


--------------------------------------------------
▶️ MEMULAI TRIAL KE-0
⚙️ Parameter: Corong=True, LR=0.0001, Batch=32, Dense=128, Dropout=0.2, Opt=RMSprop
--------------------------------------------------

   ✓ Fold 1 Selesai | Akurasi: 47.80%
   ✓ Fold 2 Selesai | Akurasi: 47.80%
   ✓ Fold 3 Selesai | Akurasi: 43.96%
   ✓ Fold 4 Selesai | Akurasi: 45.30%


[I 2026-05-13 11:45:23,934] Trial 0 finished with value: 0.4691761277396636 and parameters: {'use_corong': True, 'learning_rate': 0.0001, 'batch_size': 32, 'dense_units': 128, 'dropout_rate': 0.2, 'optimizer': 'RMSprop'}. Best is trial 0 with value: 0.4691761277396636.


   ✓ Fold 5 Selesai | Akurasi: 49.72%
🏁 KESIMPULAN TRIAL 0 | Rata-rata Akurasi: 46.92%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-1
⚙️ Parameter: Corong=False, LR=0.0001, Batch=32, Dense=128, Dropout=0.2, Opt=Adam
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 40.66%
   ✓ Fold 2 Selesai | Akurasi: 10.44%
   ✓ Fold 3 Selesai | Akurasi: 21.43%
   ✓ Fold 4 Selesai | Akurasi: 48.07%


[I 2026-05-13 12:40:22,097] Trial 1 finished with value: 0.31964058041406107 and parameters: {'use_corong': False, 'learning_rate': 0.0001, 'batch_size': 32, 'dense_units': 128, 'dropout_rate': 0.2, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.4691761277396636.


...

   ✓ Fold 5 Selesai | Akurasi: 48.62%
🏁 KESIMPULAN TRIAL 17 | Rata-rata Akurasi: 53.85%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-18
⚙️ Parameter: Corong=False, LR=0.0005, Batch=16, Dense=256, Dropout=0.4, Opt=RMSprop
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 51.10%
   ✓ Fold 2 Selesai | Akurasi: 40.11%
   ✓ Fold 3 Selesai | Akurasi: 46.70%
   ✓ Fold 4 Selesai | Akurasi: 41.99%


[I 2026-05-13 22:35:40,993] Trial 18 finished with value: 0.4603545625645073 and parameters: {'use_corong': False, 'learning_rate': 0.0005, 'batch_size': 16, 'dense_units': 256, 'dropout_rate': 0.4, 'optimizer': 'RMSprop'}. Best is trial 17 with value: 0.5385101086758545.


   ✓ Fold 5 Selesai | Akurasi: 50.28%
🏁 KESIMPULAN TRIAL 18 | Rata-rata Akurasi: 46.04%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-19
⚙️ Parameter: Corong=True, LR=5e-05, Batch=16, Dense=256, Dropout=0.4, Opt=Adam
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 46.70%
   ✓ Fold 2 Selesai | Akurasi: 37.36%
   ✓ Fold 3 Selesai | Akurasi: 48.90%
   ✓ Fold 4 Selesai | Akurasi: 43.65%


[I 2026-05-13 23:02:25,478] Trial 19 finished with value: 0.4360998117904195 and parameters: {'use_corong': True, 'learning_rate': 5e-05, 'batch_size': 16, 'dense_units': 256, 'dropout_rate': 0.4, 'optimizer': 'Adam'}. Best is trial 17 with value: 0.5385101086758545.


   ✓ Fold 5 Selesai | Akurasi: 41.44%
🏁 KESIMPULAN TRIAL 19 | Rata-rata Akurasi: 43.61%

🏆 HASIL TUNING TERBAIK DITEMUKAN!
  Akurasi Rata-rata Tertinggi: 53.85%
  Coba Parameter Ini di Script Final Anda:
    - use_corong: True
    - learning_rate: 0.0005
    - batch_size: 16
    - dense_units: 256
    - dropout_rate: 0.4
    - optimizer: Adam


In [ ]:
# import os
# import pathlib
# import numpy as np
# import pandas as pd
# import tensorflow as tf
# import optuna
# from datetime import datetime

# from tensorflow.keras import layers, models
# from tensorflow.keras.optimizers import Adam,RMSprop
# from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# from sklearn.metrics import accuracy_score
# from sklearn.model_selection import StratifiedKFold
# from sklearn.utils import class_weight

# # --- KONFIGURASI FOLDER KHUSUS TUNING ---
# DATASET_DIR = pathlib.Path("data_cropped") # Tetap membaca dari sumber yang sama
# TUNING_DIR = "HASIL_TUNING_OPTUNA"
# LOG_FILE = os.path.join(TUNING_DIR, "log_eksperimen_tuning.csv")
# IMG_SIZE = 224

# # Buat folder khusus tuning jika belum ada
# os.makedirs(TUNING_DIR, exist_ok=True)

# # Pemuatan Path Data
# all_image_paths = [str(p) for p in DATASET_DIR.glob('*/*.png')]
# class_names_list = sorted([item.name for item in DATASET_DIR.glob('*') if item.is_dir()])
# label_to_index = dict((name, index) for index, name in enumerate(class_names_list))
# all_image_labels = [label_to_index[pathlib.Path(p).parent.name] for p in all_image_paths]

# X = np.array(all_image_paths)
# y = np.array(all_image_labels)

# # --- FUNGSI PIPELINE DATA ---
# def process_path(file_path, label):
#     img = tf.io.read_file(file_path)
#     img = tf.io.decode_image(img, channels=4, expand_animations=False)
#     img = tf.cast(img, tf.float32)
#     rgb = img[..., :3]
#     alpha = img[..., 3:4] / 255.0
#     img_fixed = rgb * alpha
#     img_resized = tf.image.resize(img_fixed, [IMG_SIZE, IMG_SIZE])
#     img_final = img_resized / 255.0
#     img_final.set_shape([IMG_SIZE, IMG_SIZE, 3])
#     return img_final, tf.one_hot(label, depth=len(class_names_list))

# # --- FUNGSI BUILD MODEL ---
# def build_model(dense_units, dropout_rate):
#     inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
#     x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
    
#     max_p = layers.GlobalMaxPooling2D()(x)
#     avg_p = layers.GlobalAveragePooling2D()(x)
#     merged = layers.Concatenate()([max_p, avg_p])
    
#     x = layers.Dense(dense_units, activation='relu')(merged)
#     x = layers.Dropout(dropout_rate)(x)
#     outputs = layers.Dense(4, activation='softmax')(x)
    
#     return models.Model(inputs=inputs, outputs=outputs)

# # --- FUNGSI TUJUAN OPTUNA (OBJECTIVE) ---
# # --- FUNGSI TUJUAN OPTUNA (OBJECTIVE) ---
# def objective(trial):
#     # 1. Optuna Memilih Angka Kombinasi
#     lr = trial.suggest_categorical("learning_rate", [5e-5, 1e-4, 5e-4])
#     batch_size = trial.suggest_categorical("batch_size", [16, 32])
#     dense_units = trial.suggest_categorical("dense_units", [128, 256])
#     dropout_rate = trial.suggest_categorical("dropout_rate", [0.2, 0.3, 0.4])
#     optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])

#     # --- TAMBAHAN PRINT INFO TRIAL ---
#     print(f"\n" + "-"*50)
#     print(f"▶️ MEMULAI TRIAL KE-{trial.number}")
#     print(f"⚙️ Parameter: LR={lr}, Batch={batch_size}, Dense={dense_units}, Dropout={dropout_rate}, Optimizer={optimizer_name}")
#     print("-"*50)

#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     fold_accs = []

#     # 2. Uji Kombinasi ke dalam 5 Lipatan (K-Fold)
#     for fold_no, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#         X_train, X_val = X[train_idx], X[val_idx]
#         y_train, y_val = y[train_idx], y[val_idx]

#         weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
#         alpha_list = [float(w) for w in weights]

#         train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).map(process_path).shuffle(len(X_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
#         val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).map(process_path).batch(batch_size).prefetch(tf.data.AUTOTUNE)

#         tf.keras.backend.clear_session()
#         model = build_model(dense_units, dropout_rate)
        
#         if optimizer_name == "Adam":
#             opt = Adam(learning_rate=lr)
#         else:            
#             opt = RMSprop(learning_rate=lr)

#         model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

#         callbacks = [
#             EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True), 
#             ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6) 
#         ]

#         model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks, verbose=0)
        
#         y_val_true = np.concatenate([lbl for img, lbl in val_ds], axis=0)
#         y_val_pred = model.predict(val_ds, verbose=0)
#         acc = accuracy_score(np.argmax(y_val_true, axis=1), np.argmax(y_val_pred, axis=1))
#         fold_accs.append(acc)
        
#         # --- TAMBAHAN PRINT INFO PER FOLD ---
#         print(f"   ✓ Fold {fold_no} Selesai | Akurasi: {acc*100:.2f}%")

#     # 3. Dapatkan Rata-rata Akurasi dari 5 Lipatan
#     avg_accuracy = np.mean(fold_accs)
    
#     # --- TAMBAHAN PRINT KESIMPULAN TRIAL ---
#     print(f"🏁 KESIMPULAN TRIAL {trial.number} | Rata-rata Akurasi: {avg_accuracy*100:.2f}%")
    
#     # --- PENCATATAN LOG KE CSV ---
#     log_data = {
#         "Trial_No": trial.number,
#         "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#         "Learning_Rate": lr,
#         "Batch_Size": batch_size,
#         "Dense_Units": dense_units,
#         "Dropout_Rate": dropout_rate,
#         "Optimizer": optimizer_name,
#         "Avg_Accuracy": round(avg_accuracy * 100, 2)
#     }
#     df_log = pd.DataFrame([log_data])
#     if not os.path.exists(LOG_FILE):
#         df_log.to_csv(LOG_FILE, index=False)
#     else:
#         df_log.to_csv(LOG_FILE, mode='a', header=False, index=False)

#     return avg_accuracy

# # --- EKSEKUSI TUNING ---
# print("\n[INFO] Memulai Proses Hyperparameter Tuning...")
# print(f"[INFO] Hasil akan dicatat secara real-time di: {LOG_FILE}\n")

# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=20)

# print("\n" + "="*60)
# print("🏆 HASIL TUNING TERBAIK DITEMUKAN!")
# print(f"  Akurasi Rata-rata Tertinggi: {study.best_value*100:.2f}%")
# print(f"  Coba Parameter Ini di Script Final Anda:")
# for key, value in study.best_params.items():
#     print(f"    - {key}: {value}")
# print("="*60)

[I 2026-04-14 23:49:31,967] A new study created in memory with name: no-name-22752cc5-d35d-4456-b75f-5fba88ca35ec



[INFO] Memulai Proses Hyperparameter Tuning...
[INFO] Hasil akan dicatat secara real-time di: HASIL_TUNING_OPTUNA\log_eksperimen_tuning.csv


--------------------------------------------------
▶️ MEMULAI TRIAL KE-0
⚙️ Parameter: LR=0.0005, Batch=16, Dense=256, Dropout=0.2, Optimizer=Adam
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 52.49%
   ✓ Fold 2 Selesai | Akurasi: 58.01%
   ✓ Fold 3 Selesai | Akurasi: 48.62%
   ✓ Fold 4 Selesai | Akurasi: 57.46%


[I 2026-04-15 00:16:17,715] Trial 0 finished with value: 0.5364825046040516 and parameters: {'learning_rate': 0.0005, 'batch_size': 16, 'dense_units': 256, 'dropout_rate': 0.2, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.5364825046040516.


   ✓ Fold 5 Selesai | Akurasi: 51.67%
🏁 KESIMPULAN TRIAL 0 | Rata-rata Akurasi: 53.65%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-1
⚙️ Parameter: LR=5e-05, Batch=16, Dense=128, Dropout=0.3, Optimizer=RMSprop
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 41.99%
   ✓ Fold 2 Selesai | Akurasi: 45.86%
   ✓ Fold 3 Selesai | Akurasi: 40.33%
   ✓ Fold 4 Selesai | Akurasi: 49.17%


[I 2026-04-15 00:42:15,459] Trial 1 finished with value: 0.44136279926335176 and parameters: {'learning_rate': 5e-05, 'batch_size': 16, 'dense_units': 128, 'dropout_rate': 0.3, 'optimizer': 'RMSprop'}. Best is trial 0 with value: 0.5364825046040516.


...

   ✓ Fold 5 Selesai | Akurasi: 48.89%
🏁 KESIMPULAN TRIAL 17 | Rata-rata Akurasi: 53.20%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-18
⚙️ Parameter: LR=0.0005, Batch=32, Dense=256, Dropout=0.2, Optimizer=Adam
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 48.62%
   ✓ Fold 2 Selesai | Akurasi: 60.77%
   ✓ Fold 3 Selesai | Akurasi: 53.04%
   ✓ Fold 4 Selesai | Akurasi: 50.28%


[I 2026-04-15 09:07:09,643] Trial 18 finished with value: 0.5331921424186618 and parameters: {'learning_rate': 0.0005, 'batch_size': 32, 'dense_units': 256, 'dropout_rate': 0.2, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.5364825046040516.


   ✓ Fold 5 Selesai | Akurasi: 53.89%
🏁 KESIMPULAN TRIAL 18 | Rata-rata Akurasi: 53.32%

--------------------------------------------------
▶️ MEMULAI TRIAL KE-19
⚙️ Parameter: LR=0.0005, Batch=32, Dense=256, Dropout=0.2, Optimizer=Adam
--------------------------------------------------
   ✓ Fold 1 Selesai | Akurasi: 56.91%
   ✓ Fold 2 Selesai | Akurasi: 50.83%
   ✓ Fold 3 Selesai | Akurasi: 51.93%
   ✓ Fold 4 Selesai | Akurasi: 51.38%


[I 2026-04-15 09:43:18,484] Trial 19 finished with value: 0.5376550030693676 and parameters: {'learning_rate': 0.0005, 'batch_size': 32, 'dense_units': 256, 'dropout_rate': 0.2, 'optimizer': 'Adam'}. Best is trial 19 with value: 0.5376550030693676.


   ✓ Fold 5 Selesai | Akurasi: 57.78%
🏁 KESIMPULAN TRIAL 19 | Rata-rata Akurasi: 53.77%

🏆 HASIL TUNING TERBAIK DITEMUKAN!
  Akurasi Rata-rata Tertinggi: 53.77%
  Coba Parameter Ini di Script Final Anda:
    - learning_rate: 0.0005
    - batch_size: 32
    - dense_units: 256
    - dropout_rate: 0.2
    - optimizer: Adam
